# UPE Data – Exploratory Data Analysis

This notebook contains two parts:

| Part | Focus |
|------|-------|
| **A – Time-series analysis** | How do photon counts evolve over time? |
| **B – Count distribution analysis** | What is the statistical shape of the photon count distribution? |

In [13]:
"""
Interactive UPE (ultra-weak photon emission) detection-rate analysis.
Loads tab-separated photon-count exports from a folder, computes a range of
averages (per file, per channel, per hand, palmar/dorsal, per measurement,
overall), calculates absolute and percentage differences across specific 
experimental phases, and plots them with an ipywidgets control panel.
"""

import os
import warnings
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# IPython & ipywidgets imports
from IPython.display import display
from IPython.display import HTML as DisplayHTML 
from ipywidgets import (
    HBox, HTML, Checkbox, IntSlider, Layout, VBox, fixed, interactive_output,
)

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
DATA_FOLDER = r'c:\Users\Gebruiker\Coderen\Project\proefmap\Echte_data' 
FILE_EXTENSION = '.txt'
ALL_CHANNELS = ['LD', 'LP', 'RP', 'RD']

LEFT_CHANNELS = ('LD', 'LP')
RIGHT_CHANNELS = ('RP', 'RD')

LEFT_KEYWORDS = ('links', 'left')
RIGHT_KEYWORDS = ('rechts', 'right')

BG_HEADER_SIGNALS = ('Photon counts - Left', 'Photon counts - Right')
BG_USECOLS = [1, 3]
BG_COL_NAMES = ['BG_Left', 'BG_Right']

BIN_ROWS = 20 

PHASE_BASELINE_END = 120  
PHASE_INTERVENTION_END = 300  
PHASE_RECOVERY_END = 600  

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def _is_background_file(file_path: str) -> bool:
    try:
        with open(file_path, encoding='utf-8', errors='replace') as f:
            header = f.readline()
        return any(sig in header for sig in BG_HEADER_SIGNALS)
    except OSError:
        return False

def _infer_channels(file_name: str, file_path: str) -> list[str] | None:
    name = file_name.lower()
    if any(k in name for k in LEFT_KEYWORDS):
        return list(LEFT_CHANNELS)
    if any(k in name for k in RIGHT_KEYWORDS):
        return list(RIGHT_CHANNELS)

    try:
        with open(file_path, encoding='utf-8', errors='replace') as f:
            header = f.readline()
        if any(ch in header for ch in LEFT_CHANNELS):
            return list(LEFT_CHANNELS)
        if any(ch in header for ch in RIGHT_CHANNELS):
            return list(RIGHT_CHANNELS)
    except OSError as exc:
        warnings.warn(f"Could not read header of '{file_name}': {exc}")

    warnings.warn(f"Could not infer channels for '{file_name}'. Skipped.")
    return None

def _group_avg(all_df: pd.DataFrame, grouper) -> pd.DataFrame:
    avg = all_df.T.groupby(grouper).mean().T
    avg.columns = [f"{c}_Avg" for c in avg.columns]
    return avg.dropna(axis=1, how='all')

def _bin_to_cps(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    n = (len(data) // BIN_ROWS) * BIN_ROWS
    binned = data.iloc[:n].groupby(np.arange(n) // BIN_ROWS).sum().reset_index(drop=True)
    return binned, pd.Series(np.arange(len(binned)), dtype=float)

def _safe_column_concat(frames: list[pd.DataFrame], keys: list[str]) -> pd.DataFrame:
    max_len = max(len(f) for f in frames)
    padded = [f.reindex(range(max_len)) for f in frames]
    return pd.concat(padded, keys=keys, axis=1)

def calc_robust_pct(numerator, denominator):
    numerator = pd.Series(numerator)
    denominator = pd.Series(denominator)

    denominator = denominator.replace(0, np.nan)

    return ((numerator - denominator) / denominator) * 100

def calc_robust_pct(male, female):
    female = female.replace(0, np.nan)
    return ((male - female) / female) * 100

def _build_average_frames(all_df: pd.DataFrame) -> tuple[list[pd.DataFrame], list[str], dict]:
    avg_per_file = all_df.T.groupby(level=0).mean().T
    overall_avg = avg_per_file.mean(axis=1).rename('Overall_Avg')

    sex_level = all_df.columns.get_level_values("sex")
    channel_level = all_df.columns.get_level_values("channel")
    
    hand_grouper = ['Left' if ch in LEFT_CHANNELS else 'Right' for ch in channel_level]
    
    sex_avg = _group_avg(all_df, sex_level)
    hand_avg = _group_avg(all_df, hand_grouper)
    channel_avg = _group_avg(all_df, channel_level)

    sex_channel_frames = {}
    for sex in ["male", "female"]:
        for channel in ["LP", "RP", "LD", "RD"]:

            mask = (
                (sex_level == sex)
                & (channel_level == channel)
            )

            col_name = f"{channel}_{sex}_Avg"

            if mask.any():
                sex_channel_frames[col_name] = all_df.loc[:, mask].mean(axis=1)
            else:
                sex_channel_frames[col_name] = pd.Series(
                    np.nan,
                    index=all_df.index
                )

    sex_channel_df = pd.DataFrame(sex_channel_frames)
    sex_channel_df["Left_male_Avg"] = sex_channel_df[
        ["LP_male_Avg", "LD_male_Avg"]
    ].mean(axis=1)

    sex_channel_df["Right_male_Avg"] = sex_channel_df[
        ["RP_male_Avg", "RD_male_Avg"]
    ].mean(axis=1)

    sex_channel_df["Palmar_male_Avg"] = sex_channel_df[
        ["LP_male_Avg", "RP_male_Avg"]
    ].mean(axis=1)

    sex_channel_df["Dorsal_male_Avg"] = sex_channel_df[
        ["LD_male_Avg", "RD_male_Avg"]
    ].mean(axis=1)

    sex_channel_df["Left_female_Avg"] = sex_channel_df[
        ["LP_female_Avg", "LD_female_Avg"]
    ].mean(axis=1)

    sex_channel_df["Right_female_Avg"] = sex_channel_df[
        ["RP_female_Avg", "RD_female_Avg"]
    ].mean(axis=1)

    sex_channel_df["Palmar_female_Avg"] = sex_channel_df[
        ["LP_female_Avg", "RP_female_Avg"]
    ].mean(axis=1)

    sex_channel_df["Dorsal_female_Avg"] = sex_channel_df[
        ["LD_female_Avg", "RD_female_Avg"]
    ].mean(axis=1)

    palmar_mask = channel_level.isin(['LP', 'RP'])
    dorsal_mask = channel_level.isin(['LD', 'RD'])
    palmar_avg = all_df.loc[:, palmar_mask].mean(axis=1).rename('Palmar_Avg (LP+RP)')
    dorsal_avg = all_df.loc[:, dorsal_mask].mean(axis=1).rename('Dorsal_Avg (LD+RD)')

    # Vluchtige verschillen (voor in de grafiek)
    diff_avg = (palmar_avg - dorsal_avg).rename('Verschil (Palmair - Dorsaal)')
    pct_diff_avg = ((palmar_avg - dorsal_avg) / dorsal_avg.replace(0, np.nan) * 100).rename('Procentueel_Verschil (%)')

    ld_avg = all_df.loc[:, channel_level == 'LD'].mean(axis=1) if 'LD' in channel_level else pd.Series(np.nan, index=all_df.index)
    lp_avg = all_df.loc[:, channel_level == 'LP'].mean(axis=1) if 'LP' in channel_level else pd.Series(np.nan, index=all_df.index)
    rp_avg = all_df.loc[:, channel_level == 'RP'].mean(axis=1) if 'RP' in channel_level else pd.Series(np.nan, index=all_df.index)
    rd_avg = all_df.loc[:, channel_level == 'RD'].mean(axis=1) if 'RD' in channel_level else pd.Series(np.nan, index=all_df.index)

    pct_diff_lp_rp = ((lp_avg - rp_avg) / rp_avg.replace(0, np.nan) * 100).rename('Procentueel_Verschil (LP vs RP) (%)')
    pct_diff_ld_rd = ((ld_avg - rd_avg) / rd_avg.replace(0, np.nan) * 100).rename('Procentueel_Verschil (LD vs RD) (%)')

    meas_parts = {}
    for i in range(1, 5):
        cols = [c for c in avg_per_file.columns if f'_m{i}_' in c]
        if cols:
            meas_parts[f'm{i}_Avg'] = avg_per_file[cols].mean(axis=1)
    meas_avg = pd.DataFrame(meas_parts)

    indiv_df = all_df.copy()
    indiv_df.columns = [
        "_".join(str(v) for v in col if pd.notna(v))
        for col in indiv_df.columns
    ]
    indiv_df.dropna(axis=1, how='all', inplace=True)

    sex_diff_df = pd.DataFrame(index=all_df.index)

    comparisons = {
        "LP": ("LP_male_Avg", "LP_female_Avg"),
        "RP": ("RP_male_Avg", "RP_female_Avg"),
        "LD": ("LD_male_Avg", "LD_female_Avg"),
        "RD": ("RD_male_Avg", "RD_female_Avg"),
        "Links": ("Left_male_Avg", "Left_female_Avg"),
        "Rechts": ("Right_male_Avg", "Right_female_Avg"),
        "Palmair": ("Palmar_male_Avg", "Palmar_female_Avg"),
        "Dorsaal": ("Dorsal_male_Avg", "Dorsal_female_Avg"),
    }

    for label, (male_col, female_col) in comparisons.items():
        sex_diff_df[f"{label}_MV_pct"] = calc_robust_pct(
            sex_channel_df[male_col],
            sex_channel_df[female_col]
        )

    parts = [
    indiv_df,
    avg_per_file,
    overall_avg,
    hand_avg,
    sex_avg,
    channel_avg,
    sex_channel_df,
    sex_diff_df,
    palmar_avg,
    dorsal_avg,
    diff_avg,
    pct_diff_avg,
    pct_diff_lp_rp,
    pct_diff_ld_rd,
]
    if not meas_avg.empty:
        parts.append(meas_avg)

    plot_groups = {
        'hand': list(hand_avg.columns),
        'channel': list(channel_avg.columns),
        'sex': list(sex_avg.columns),
        'sex_channel': list(sex_channel_df.columns),
        'sex_diff': list(sex_diff_df.columns),
    }
    return parts, list(indiv_df.columns), plot_groups

SEX_LOOKUP = {
    "p1" : "male",
    "p2" : "female",
    "p3" : "male",
    "p4" : "female",
    "p5" : "male",
    "p6" : "male",
    "p7" : "male",
    "p8" : "male",
    "p9" : "female",
    "p10" : "male",
    "p11" : "male",
    "p12" : "female",
    "p13" : "female",
    "p14" : "female",
    "p15" : "male",

}

def extract_participant(file_name: str) -> str | None:
    match = re.search(
        r"exercise_(p\d+)_",
        file_name.lower()
    )

    if match:
        return match.group(1)

    return None

# ---------------------------------------------------------
# Data loading
# ---------------------------------------------------------
def load_and_process_data(folder_path: str):
    if not os.path.isdir(folder_path):
        print(f"Error: directory not found — '{folder_path}'")
        return pd.DataFrame(), [], {}, []

    frames, labels, bg_cols = [], [], {}
    column_metadata = []

    sex_lookup = SEX_LOOKUP

    for file_name in sorted(os.listdir(folder_path)):
        if not file_name.endswith(FILE_EXTENSION):
            continue
        path = os.path.join(folder_path, file_name)
        stem = os.path.splitext(file_name)[0]

        if _is_background_file(path):
            try:
                bg = pd.read_csv(path, sep="\t", skiprows=1, header=None, usecols=BG_USECOLS)
                bg.columns = BG_COL_NAMES
                bg_cols[f"{stem}_BG_Left"] = bg['BG_Left']
                bg_cols[f"{stem}_BG_Right"] = bg['BG_Right']
            except Exception as exc:
                warnings.warn(f"Skipping background '{file_name}': {exc}")
            continue

        channels = _infer_channels(file_name, path)
        if channels is None:
            continue

        try:
            df = pd.read_csv(path, sep="\t", skiprows=1, header=None, usecols=[1, 3])
            df.columns = channels
            df = df.reindex(columns=ALL_CHANNELS, fill_value=np.nan)

            participant = extract_participant(stem)

            if participant is None:
                warnings.warn(
                    f"Kon participant niet bepalen uit '{stem}'"
                )
                participant = "unknown"

            sex = sex_lookup.get(participant)

            if sex is None:
                warnings.warn(
                    f"Geen geslacht gevonden voor '{participant}'"
                )
                sex = "unknown"

            frames.append(df)
            labels.append(stem)

            for channel in ALL_CHANNELS:
                column_metadata.append(
                    {
                        "file": stem,
                        "participant": participant,
                        "sex": sex,
                        "channel": channel
                    }
    )
        except Exception as exc:
            warnings.warn(f"Skipping '{file_name}': {exc}")

    if not frames and not bg_cols:
        return pd.DataFrame(), [], {}, []

    if frames:
        all_df = _safe_column_concat(frames, labels)

        all_df.columns = pd.MultiIndex.from_tuples(
            [
                (
                    meta["file"],
                    meta["participant"],
                    meta["sex"],
                    meta["channel"]
                )
                for meta in column_metadata
            ],
            names=[
                "file",
                "participant",
                "sex",
                "channel"
            ]
        )
        parts, indiv_labels, plot_groups = _build_average_frames(all_df)
        processed_df = pd.concat(parts, axis=1)
    else:
        processed_df, indiv_labels, plot_groups = pd.DataFrame(), [], {}

    if bg_cols:
        bg_frame = pd.DataFrame(bg_cols)
        if processed_df.empty:
            processed_df = bg_frame
        else:
            max_len = max(len(processed_df), len(bg_frame))
            processed_df = pd.concat(
                [processed_df.reindex(range(max_len)), bg_frame.reindex(range(max_len))],
                axis=1,
            )
        indiv_labels += list(bg_cols.keys())

    processed_df.insert(0, "Time", np.arange(len(processed_df), dtype=float))
    return processed_df, labels, plot_groups, indiv_labels

# ---------------------------------------------------------
# Plotting & Stats
# ---------------------------------------------------------
def plot_combined_data(
    df, file_labels, plot_groups, individual_labels,
    window, counts_per_sec, demean, start_at_zero, 
    show_all_individual, show_individual, show_hand,
    show_ld, show_lp, show_rp, show_rd,
    show_overall, show_palmar_dorsal, show_measurement,
    show_diff, show_sex, show_pct_male_female, show_pct_diff, show_pct_lp_rp, show_pct_ld_rd
):
    if df.empty:
        return

    data = df.drop(columns="Time")
    if counts_per_sec:
        data, _ = _bin_to_cps(data)
        x = pd.Series(np.arange(len(data)), dtype=float)
        x_title, y_title = "Tijd (seconden)", "Detectie ratio (cps)"
        t_base = PHASE_BASELINE_END
        t_int = PHASE_INTERVENTION_END
        t_rec = PHASE_RECOVERY_END
    else:
        x = df["Time"]
        x_title, y_title = "Tijd index (50 ms)", "Detectie ratio (counts / 50 ms)"
        t_base = PHASE_BASELINE_END * BIN_ROWS
        t_int = PHASE_INTERVENTION_END * BIN_ROWS
        t_rec = PHASE_RECOVERY_END * BIN_ROWS

    smoothed = data.rolling(window=window, center=True, min_periods=1).mean()
    if demean:
        smoothed -= smoothed.mean()
    if start_at_zero:
        smoothed -= smoothed.iloc[0]

    C = px.colors.qualitative
    
    plot_config = [
        (show_all_individual, individual_labels, 1.0, C.Alphabet),
        (show_individual, file_labels, 1.5, C.Plotly),
        (show_hand, plot_groups.get('hand', []), 3.0, C.Plotly),
        (show_sex, plot_groups.get('sex', []), 4.0, C.Safe),
        (show_sex, plot_groups.get('sex_channel', []), 3.0, C.Safe),
        (show_pct_male_female, plot_groups.get('sex_diff', []), 3.5, ['#8c564b']),
        (show_ld, ['LD_Avg'], 3.0, [C.D3[0]]),
        (show_lp, ['LP_Avg'], 3.0, [C.D3[1]]),
        (show_rp, ['RP_Avg'], 3.0, [C.D3[2]]),
        (show_rd, ['RD_Avg'], 3.0, [C.D3[3]]),
        (show_overall, ['Overall_Avg'], 4.0, ['black']),
        (show_palmar_dorsal, ['Palmar_Avg (LP+RP)', 'Dorsal_Avg (LD+RD)'], 3.0, C.T10),
        (show_measurement, [f'm{i}_Avg' for i in range(1, 5)], 3.0, C.Vivid),
        (show_diff, ['Verschil (Palmair - Dorsaal)'], 3.5, ['#ff00ff']),
        (show_pct_diff, ['Procentueel_Verschil (%)'], 3.5, ['#00ffcc']),
        (show_pct_lp_rp, ['Procentueel_Verschil (LP vs RP) (%)'], 3.5, ['#e377c2']),
        (show_pct_ld_rd, ['Procentueel_Verschil (LD vs RD) (%)'], 3.5, ['#bcbd22']),
    ]

    fig = go.Figure()
    
    x_max = int(x.iloc[-1]) if not x.empty else 0
    
    fig.add_vrect(x0=0, x1=min(t_base, x_max), fillcolor="rgba(0, 255, 0, 0.05)", 
                  layer="below", line_width=0, annotation_text="Baseline (0-2m)", annotation_position="top left")
    if x_max > t_base:
        fig.add_vrect(x0=t_base, x1=min(t_int, x_max), fillcolor="rgba(255, 165, 0, 0.08)", 
                      layer="below", line_width=0, annotation_text="Interventie (2-5m)", annotation_position="top left")
    if x_max > t_int:
        fig.add_vrect(x0=t_int, x1=min(t_rec, x_max), fillcolor="rgba(0, 0, 255, 0.05)", 
                      layer="below", line_width=0, annotation_text="Herstel (5-10m)", annotation_position="top left")

    for show, col_labels, lw, palette in plot_config:
        if not show:
            continue
        for i, label in enumerate(col_labels):
            if label not in smoothed.columns:
                continue
            
            # Subtiele transparantie voor de vluchtige ruis-lijnen
            if label in individual_labels:
                opacity = 0.15
                width = 1.0
            else:
                opacity = 0.4 if "pct" in label.lower() else 1.0
                width = lw
                        
            fig.add_trace(go.Scatter(
                x=x, y=smoothed[label], mode='lines', name=label,
                line=dict(color=palette[i % len(palette)], width=width),
                opacity=opacity,
                hovertemplate=(f"<b>{label}</b><br>Tijd: %{{x}}<br>Waarde: %{{y:.2f}}<extra></extra>"),
            ))

    fig.update_layout(
        title_text="<b>UPE Detectie & Fase Analyse</b>",
        title_font_size=22,
        xaxis_title=x_title,
        yaxis_title=y_title,
        plot_bgcolor='white',
        height=750,
        hovermode="x unified",
        font=dict(size=13),
        legend=dict(bgcolor='rgba(255,255,255,0.85)', bordercolor='lightgray', borderwidth=1, font_size=11),
        xaxis=dict(showgrid=False, linecolor='black', mirror=True),
        yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', mirror=True),
        margin=dict(t=60, b=60, l=70, r=20),
    )
    
    fig.show()

    # --- NIEUW: Robuuste Tabel Generatie ---
    # We voegen de fases toe aan onze data
    df_stats = smoothed.copy()
    df_stats['Fase'] = 'Onbekend'
    df_stats.loc[x < t_base, 'Fase'] = '1. Baseline (0-2m)'
    df_stats.loc[(x >= t_base) & (x < t_int), 'Fase'] = '2. Interventie (2-5m)'
    df_stats.loc[(x >= t_int) & (x <= t_rec), 'Fase'] = '3. Herstel (5-10m)'
    
    # 1. We berekenen uitsluitend de absolute gemiddeldes over de hele fase
    summary = df_stats.groupby('Fase').mean()

    rename_map = {
        "LP_MV_pct": "LP man-vrouw (%)",
        "RP_MV_pct": "RP man-vrouw (%)",
        "LD_MV_pct": "LD man-vrouw (%)",
        "RD_MV_pct": "RD man-vrouw (%)",
        "Links_MV_pct": "Links man-vrouw (%)",
        "Rechts_MV_pct": "Rechts man-vrouw (%)",
        "Palmair_MV_pct": "Palmair man-vrouw (%)",
        "Dorsaal_MV_pct": "Dorsaal man-vrouw (%)",
    }

    summary = summary.rename(columns=rename_map)

    phase_cols = {
        "LP man-vrouw (%)": "LP_MV_pct",
        "RP man-vrouw (%)": "RP_MV_pct",
        "LD man-vrouw (%)": "LD_MV_pct",
        "RD man-vrouw (%)": "RD_MV_pct",
        "Links man-vrouw (%)": "Links_MV_pct",
        "Rechts man-vrouw (%)": "Rechts_MV_pct",
        "Palmair man-vrouw (%)": "Palmair_MV_pct",
        "Dorsaal man-vrouw (%)": "Dorsaal_MV_pct",
    }

    for new_name, old_name in phase_cols.items():
        if old_name in summary.columns:
            summary[new_name] = summary[old_name]
    
    # Een veilige functie om procenten te berekenen (voorkomt deling door 0 of lege data)
    def calc_robust_pct(a, b):
        if pd.isna(a) or pd.isna(b) or b == 0:
            return np.nan
        return ((a - b) / b) * 100
    
    html_str = f"""
    <div style="font-family: Arial, sans-serif; ...">
        <h3 style="margin-top: 0; color: #333;">
            Gemiddelde Verschillen per Fase (Robuuste Methode)
        </h3>

        <table style="width: 100%; border-collapse: collapse; text-align: left;">
            <tr style="background-color: #eee; border-bottom: 2px solid #ccc;">
                <th style="padding: 8px;">Fase</th>
                <th style="padding: 8px;">Palmair - Dorsaal (Abs)</th>
                <th style="padding: 8px;">Palmair vs Dorsaal (%)</th>
                <th style="padding: 8px;">LP vs RP (%)</th>
                <th style="padding: 8px;">LD vs RD (%)</th>
                <th style="padding: 8px;">Man vs Vrouw (%)</th>
            </tr>
    """
    
    for index in summary.index:
        # Haal de absolute fase-gemiddeldes op
        pal_val = summary.get('Palmar_Avg (LP+RP)', pd.Series(dtype=float)).get(index, np.nan)
        dor_val = summary.get('Dorsal_Avg (LD+RD)', pd.Series(dtype=float)).get(index, np.nan)
        lp_val  = summary.get('LP_Avg', pd.Series(dtype=float)).get(index, np.nan)
        rp_val  = summary.get('RP_Avg', pd.Series(dtype=float)).get(index, np.nan)
        ld_val  = summary.get('LD_Avg', pd.Series(dtype=float)).get(index, np.nan)
        rd_val  = summary.get('RD_Avg', pd.Series(dtype=float)).get(index, np.nan)
        male_val = summary.get(
            'male_Avg',
            pd.Series(dtype=float)
        ).get(index, np.nan)
        female_val = summary.get('female_Avg', pd.Series(dtype=float)).get(index,np.nan)
        # 2. Bereken de wiskundig robuuste verschillen
        abs_diff = np.nan if pd.isna(pal_val) else pal_val - dor_val
        pct_pal_dor = calc_robust_pct(pal_val, dor_val)
        pct_lp_rp = calc_robust_pct(lp_val, rp_val)
        pct_ld_rd = calc_robust_pct(ld_val, rd_val)
        pct_mf = calc_robust_pct(male_val, female_val)

        # Formatteer netjes voor weergave
        f_abs = f"{abs_diff:.3f}" if not pd.isna(abs_diff) else "-"
        f_pd  = f"{pct_pal_dor:.2f} %" if not pd.isna(pct_pal_dor) else "-"
        f_lprp = f"{pct_lp_rp:.2f} %" if not pd.isna(pct_lp_rp) else "-"
        f_ldrd = f"{pct_ld_rd:.2f} %" if not pd.isna(pct_ld_rd) else "-"
        f_mf = (
            f"{pct_mf:.2f} %"
            if not pd.isna(pct_mf)
            else "-"
        )

        html_str += f"""
            <tr style="border-bottom: 1px solid #ddd;">
                <td style="padding: 8px; font-weight: bold;">{index}</td>
                <td style="padding: 8px;">{f_abs}</td>
                <td style="padding: 8px;">{f_pd}</td>
                <td style="padding: 8px; color: #d62728;">{f_lprp}</td>
                <td style="padding: 8px; color: #7f7f7f;">{f_ldrd}</td>
                <td style="padding: 8px; color: #8c564b;">{f_mf}</td>
            </tr>
        """
    html_str += "</table></div>"
    
    display(DisplayHTML(html_str)) 

# ---------------------------------------------------------
# UI helpers & Main
# ---------------------------------------------------------
def _section(title: str):
    return HTML(f"<div style='font-size:12px; font-weight:600; text-transform:uppercase; letter-spacing:0.06em; color:#666; margin:8px 0 2px;'>{title}</div>")

def _cb(desc: str, val: bool = False):
    return Checkbox(value=val, description=desc, style={'description_width': 'initial'}, layout=Layout(width='280px'))

_wide = Layout(width='420px')

processed_df, file_labels, plot_groups, individual_labels = load_and_process_data(DATA_FOLDER)

_n_bg = sum('_BG_' in c for c in individual_labels)
if _n_bg: print(f"✓ Loaded {_n_bg} background channel(s).")
if file_labels: print(f"✓ Loaded {len(file_labels)} hand-measurement file(s).")

slider = IntSlider(min=1, max=1001, step=10, value=101, description="Window:", style={'description_width': '60px'}, layout=_wide)

counts_per_sec_cb = _cb("Counts per second (cps)", val=False)
demean_cb = _cb("Demean recordings")
start_at_zero_cb = _cb("Start all at y = 0")

show_all_indiv_cb = _cb("All individual recordings", val=False)
show_indiv_cb = _cb("Per-file averages")
show_hand_cb = _cb("Per-hand averages")
show_pal_dors_cb = _cb("Palmar & dorsal averages", val=True)
show_meas_cb = _cb("Per-measurement averages")
show_overall_cb = _cb("Overall average")

show_diff_cb = _cb("Toon Verschil (Palmair - Dorsaal)", val=True)
show_pct_diff_cb = _cb("Toon Procentueel Verschil (%)", val=False)

show_ld_cb = _cb("Toon LD (Links Dorsaal)", val=False)
show_lp_cb = _cb("Toon LP (Links Palmair)", val=False)
show_rp_cb = _cb("Toon RP (Rechts Palmair)", val=False)
show_rd_cb = _cb("Toon RD (Rechts Dorsaal)", val=False)

show_pct_lp_rp_cb = _cb("Toon % Verschil (LP vs RP)", val=False)
show_pct_ld_rd_cb = _cb("Toon % Verschil (LD vs RD)", val=False)

show_sex_cb = _cb(
    "Toon mannen vs vrouwen", 
    val= True
)

show_pct_mf_cb = _cb(
    "Toon % verschil (M vs V)",
    val= False
)

ui = VBox([
    HTML("<h3 style='margin:0 0 8px; font-size:15px;'>UPE Analyse Controlepaneel</h3>"),
    _section("Smoothing"), HBox([slider]),
    _section("Tijd / Schaal"), HBox([counts_per_sec_cb, demean_cb, start_at_zero_cb]),
    _section("Analyse & Verschillen (Vlakken)"),
    HBox([show_pal_dors_cb, show_diff_cb, show_pct_diff_cb]),
    HBox([show_pct_lp_rp_cb, show_pct_ld_rd_cb]),
    HBox([show_sex_cb, show_pct_mf_cb]),
    _section("Losse Kanalen (Gemiddelden)"),
    HBox([show_ld_cb, show_lp_cb, show_rp_cb, show_rd_cb]),
    _section("Overige Traces"),
    HBox([show_all_indiv_cb, show_indiv_cb, show_hand_cb]),
    HBox([show_meas_cb, show_overall_cb]),
    HTML("<div style='height:8px'></div>"),
], layout=Layout(padding='12px', border='1px solid #ddd', border_radius='8px', width='fit-content'))

out = interactive_output(plot_combined_data, {
    'df': fixed(processed_df),
    'file_labels': fixed(file_labels),
    'plot_groups': fixed(plot_groups),
    'individual_labels': fixed(individual_labels),
    'window': slider,
    'counts_per_sec': counts_per_sec_cb,
    'demean': demean_cb,
    'start_at_zero': start_at_zero_cb,
    'show_all_individual': show_all_indiv_cb,
    'show_individual': show_indiv_cb,
    'show_hand': show_hand_cb,
    'show_ld': show_ld_cb,
    'show_lp': show_lp_cb,
    'show_rp': show_rp_cb,
    'show_rd': show_rd_cb,
    'show_overall': show_overall_cb,
    'show_palmar_dorsal': show_pal_dors_cb,
    'show_measurement': show_meas_cb,
    'show_diff': show_diff_cb,
    'show_pct_diff': show_pct_diff_cb,
    'show_pct_lp_rp': show_pct_lp_rp_cb,
    'show_pct_ld_rd': show_pct_ld_rd_cb,
    'show_sex': show_sex_cb,
    'show_pct_male_female': show_pct_mf_cb,
})

if not processed_df.empty:
    display(ui, out)
else:
    print("Execution halted: no data loaded.")

✓ Loaded 32 hand-measurement file(s).


Output()